# Water Rights Restored to the Gila River

The impacts of irrigation on vegetation health in the Gila River Valley

## Set up: import libraries

We’ll need some Python libraries to complete this workflow.

In the cell below, making sure to keep the packages in order, add packages for:

- Working with DataFrames  
- Working with GeoDataFrames  
- Making interactive plots of tabular and vector data


### Reflect and Respond #1: 
What are we using the rest of these packages for? See if you can figure it out as you complete the notebook.

In [12]:
### working with dataframes
import pandas as pd

### working with geodataframes
import geopandas as gpd

### making interactive plots of tabular and vector data
import hvplot.pandas

### work with json data
import json

### work with pattern matching rules for pathnames 
from glob import glob

### access the data we'll use for the assignment
import earthpy

### interactive plots with rasters 
import hvplot.xarray

### handle geospatial rasters
import rioxarray as rxr

### work with arrays
import xarray as xr

### import spatial reference to address basemap errors
import cartopy.crs as ccrs

## Step 1: Download sample data

In this analysis, you’ll need to download multiple data files to your computer. You’ll need to set up a folder/directory for the files, and while you’re at it, download the sample data there.

In the cell below, replace ‘Project Name’ with ‘Gila River Vegetation' and 'my-data-folder' with a **descriptive** directory name.

In [13]:
### define the project and data
project = earthpy.Project(
    'Gila River Vegetation', dirname = 'gila_data')

### get the data
project.get_data()

### Earth Data Science data formats:

| Data type | Descriptions | Common file formats | Python type |
|------------------|------------------|------------------|------------------|
| Time Series | The same data points (e.g. streamflow) collected multiple times over time | Tabular formats (e.g. .csv, or .xlsx) | pandas DataFrame |
| Vector | Points, lines, and polygons (with coordinates) | Shapefile (often an archive like a `.zip` file because a Shapefile is actually a collection of at least 3 files) | geopandas GeoDataFrame |
| Raster | Evenly spaced spatial grid (with coordinates) | GeoTIFF (`.tif`), NetCDF (`.nc`), HDF (`.hdf`) | rioxarray DataArray |

Read more about [vector](https://www.earthdatascience.org/courses/use-data-open-source-python/intro-vector-data-python/spatial-data-vector-shapefiles/) and [raster](https://earthdatascience.org/courses/intro-to-earth-data-science/file-formats/use-spatial-data/use-raster-data/) data in the Earth Data Science textbook.

### Reflect and Respond #2:
For this coding challenge, we are interested in the boundary of the Gila River Indian Community, and the health of vegetation in the area measured on a scale from -1 to 1. In the cell below, answer the following questions: What data type do you think the boundary will be? What about the vegetation health?

## Step 2: Make a site map

### Step 2A: Load the Gila River Indian Community boundary  

- Locate the Tribal Subdivision files in your download directory  
- Change `'subdivision-directory'` to the actual location
- Load the data into Python and check that it worked


In [14]:
### figure out where the download lives
print(project.project_dir)

C:\Users\FluvialGroup\AppData\Local\earth-analytics\earth-analytics\gila_data


In [15]:
### load in the boundary data
aitsn_gdf = gpd.read_file(project.project_dir / 'tl_2020_us_aitsn')

### check that it worked
aitsn_gdf

,AIANNHCE,TRSUBCE,TRSUBNS,GEOID,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,2430,653,02419073,2430653,Red Valley,Red Valley Chapter,T2,D7,G2300,A,922036695,195247,+36.6294607,-109.0550394,"POLYGON ((-109.2827 36.64644, -109.28181 36.65..."
1,2430,665,02419077,2430665,Rock Point,Rock Point Chapter,T2,D7,G2300,A,720360268,88806,+36.6598701,-109.6166836,"POLYGON ((-109.85922 36.49859, -109.85521 36.5..."
2,2430,675,02419081,2430675,Rough Rock,Rough Rock Chapter,T2,D7,G2300,A,364475668,216144,+36.3976971,-109.7695183,"POLYGON ((-109.93053 36.40672, -109.92923 36.4..."
3,2430,325,02418975,2430325,Indian Wells,Indian Wells Chapter,T2,D7,G2300,A,717835323,133795,+35.3248534,-110.0855000,"POLYGON ((-110.24222 35.36327, -110.24215 35.3..."
4,2430,355,02418983,2430355,Kayenta,Kayenta Chapter,T2,D7,G2300,A,1419241065,1982848,+36.6884391,-110.3045616,"POLYGON ((-110.56817 36.73489, -110.56603 36.7..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479,1310,100,02418907,1310100,1,District 1,28,D7,G2300,N,139902197,0,+33.0600842,-111.5806313,"POLYGON ((-111.63622 33.11798, -111.63405 33.1..."
480,4290,550,02612186,4290550,Mission Highlands,Mission Highlands,00,D7,G2300,N,6188043,0,+48.0754384,-122.2507432,"POLYGON ((-122.27579 48.07128, -122.27578 48.0..."
481,0855,400,02418941,0855400,Fort Thompson,Fort Thompson District,07,D7,G2300,N,535432708,38653364,+44.1559680,-099.4467700,"POLYGON ((-99.66452 44.25269, -99.66449 44.255..."
482,0335,300,02784108,0335300,Indian Point,Indian Point Segment,T3,D7,G2300,N,326985,0,+48.0604594,-092.8466753,"POLYGON ((-92.85187 48.05944, -92.85186 48.059..."


You might notice in this dataset that some of the names are not easily searchable. For example, the Gila River subdivisions are named “District 1-7”! So, how do we know what to search for? We recommend making an **interactive** plot of the data so that you can find the information you need:

In [16]:
### workaround to fix error associated with basemap
if not hasattr(ccrs.PlateCarree, '_original_proj4_params'):
    ccrs.PlateCarree._original_proj4_params = ccrs.PlateCarree.proj4_params

def _proj4_params_with_lon_0(self):
    params = ccrs.PlateCarree._original_proj4_params.fget(self)
    params.setdefault('lon_0', params.get('pm', 0))
    return params

ccrs.PlateCarree.proj4_params = property(_proj4_params_with_lon_0)

In [17]:
### make an interactive map
aitsn_gdf.hvplot(

    ### set basemap
    geo = True, tiles = 'EsriImagery', 

    ### set figure size
    frame_width = 500,

    ### other map parameters
    legend = False, fill_color = None, edge_color = 'white',

    ### make all the column values in the dataset visible
    hover_cols = 'all')

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]   (index,AIANNHCE,TRSUBCE,TRSUBNS,GEOID,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON)

### Reflect and Respond #3:
What column could you use to uniquely identify the subdivisions of the reservation you want to study using this interactive map? What value do you need to use to filter the `GeoDataFrame`?

AIANNHCE

Now that you have the info you need, it’s also a good idea to check the data type. Are the values in the unique identifier column you've selected some kind of **number** or an **object** like a text string? We can’t tell just by looking, which is where the `.info()` method comes in:

In [18]:
aitsn_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 484 entries, 0 to 483
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   AIANNHCE  484 non-null    str     
 1   TRSUBCE   484 non-null    str     
 2   TRSUBNS   484 non-null    str     
 3   GEOID     484 non-null    str     
 4   NAME      484 non-null    str     
 5   NAMELSAD  484 non-null    str     
 6   LSAD      484 non-null    str     
 7   CLASSFP   484 non-null    str     
 8   MTFCC     484 non-null    str     
 9   FUNCSTAT  484 non-null    str     
 10  ALAND     484 non-null    int64   
 11  AWATER    484 non-null    int64   
 12  INTPTLAT  484 non-null    str     
 13  INTPTLON  484 non-null    str     
 14  geometry  484 non-null    geometry
dtypes: geometry(1), int64(2), str(12)
memory usage: 93.7 KB


### Reflect and Respond #4:
What is the data type of the unique identifier column? How will that affect your code?

## Step 2B: Select the Gila River subdivisions and make a site map.

- Replace `identifier` with the value you found from exploring the interactive map. Make sure that you are using the correct data type!  
- Change the plot to have a web tile basemap and add a title. Change the color of the Gila River Indian Community polygon so that it's transparent

In [21]:
### select and merge the subdivisions you want
boundary_gdf = aitsn_gdf.loc[aitsn_gdf.AIANNHCE == '1310'].dissolve()

### plot the results with web tile images and a title
### change the default color of the polygon so that you can see the basemap through it
boundary_gdf.hvplot(
    
    ### set up the background tiles
    geo = True, tiles = 'EsriImagery',

    ### fill color = see through
    fill_color = None,

    ### outline polygon
    line_color = 'black',

    ### title
    title = 'Gila River Indian Community',

    ### set size
    frame_width = 500
    
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

## Wrap up Steps 1-2

Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

In [20]:
%store boundary_gdf project

Stored 'boundary_gdf' (GeoDataFrame)
Stored 'project' (Project)


Finally, be sure to `Restart` and `Run all` to make sure your notebook works all the way through!